# Exploración CSV Raw de MiBici

Exploracion de datos para decisiones de limpieza

In [1]:
import pandas as pd
from mi_bici.config import COLUMN_PATTERNS_REGEX, URLS_VIAJES
from mi_bici.dataset import clean_data_travels, load_data_travels, load_data_from_url

print(f"Total de periodos en URLS_VIAJES: {len(URLS_VIAJES)}")
print("Periodos disponibles:", list(URLS_VIAJES.keys()))

2026-10-17 22:51:01.680 | INFO     | mi_bici.config:<module>:12 - PROJ_ROOT path is: /home/miguel/Maestria/Trimestre4/mlops/TC5061-Proyecto-Equipo03


Total de periodos en URLS_VIAJES: 12
Periodos disponibles: ['2025-10', '2025-11', '2025-12', '2026-01', '2026-02', '2026-03', '2026-04', '2026-05', '2026-06', '2026-07', '2026-08', '2026-09']


## 1. Carga y exploración base de un CSV inicial (Periodo 2025-10)

Utilizamos la función modular `load_data_travels(period="2025-10")` para cargar directamente el CSV.

In [2]:
df_2025_10 = load_data_travels(period="2025-10")
df_2025_10.info()

2026-10-17 22:51:01.714 | INFO     | mi_bici.dataset:load_data_travels:161 - Cargando archivo: datos_abiertos_2025_10.csv
<class 'pandas.DataFrame'>
RangeIndex: 420064 entries, 0 to 420063
Data columns (total 8 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   Viaje_Id            420064 non-null  int64  
 1   Usuario_Id          420064 non-null  int64  
 2   Genero              415721 non-null  str    
 3   AÃ±o_de_nacimiento  420031 non-null  float64
 4   Inicio_del_viaje    420064 non-null  str    
 5   Fin_del_viaje       420064 non-null  str    
 6   Origen_Id           420064 non-null  int64  
 7   Destino_Id          420064 non-null  int64  
dtypes: float64(1), int64(4), str(3)
memory usage: 41.3 MB


In [3]:
df_2026_08 = load_data_travels(period="2026-08")
df_2026_08.info()

2026-10-17 22:51:01.972 | INFO     | mi_bici.dataset:load_data_travels:161 - Cargando archivo: datos_abiertos_2026_08.csv
<class 'pandas.DataFrame'>
RangeIndex: 393524 entries, 0 to 393523
Data columns (total 8 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   Viaje_Id            393524 non-null  int64  
 1   Usuario_Id          393524 non-null  int64  
 2   Genero              391192 non-null  str    
 3   AÃ±o_de_nacimiento  393473 non-null  float64
 4   Inicio_del_viaje    393524 non-null  str    
 5   Fin_del_viaje       393524 non-null  str    
 6   Origen_Id           393524 non-null  int64  
 7   Destino_Id          393524 non-null  int64  
dtypes: float64(1), int64(4), str(3)
memory usage: 38.7 MB


In [ ]:
df_2025_10.head(10)

In [ ]:
df_2026_08.head(10)

## 2. Justificación empírica de las reglas de limpieza

A continuación analizamos las anomalías observadas en los CSV raw para justificar cada paso del pipeline de limpieza.

### Justificación 1: Inconsistencia en nombres de columnas y codificación (encoding)
- En algunos meses la columna de año de nacimiento viene codificada como `AÃ±o_de_nacimiento` (por decodificación ISO-8859-1 sobre texto UTF-8) y en otros como `Año_de_nacimiento`.
- Por ello se definió el diccionario de expresiones regulares `COLUMN_PATTERNS_REGEX` para mapear dinámicamente las columnas a nombres estandarizados sin depender de la codificación exacta.

In [6]:
df_2026_01 = load_data_travels(period="2026-01")
print("Columnas en 2025-10:", df_2025_10.columns.tolist())
print("Columnas en 2026-01:", df_2026_01.columns.tolist())
print("\nPatrones regex de normalización:", COLUMN_PATTERNS_REGEX)

2026-10-17 22:51:02.202 | INFO     | mi_bici.dataset:load_data_travels:161 - Cargando archivo: datos_abiertos_2026_01.csv
Columnas en 2025-10: ['Viaje_Id', 'Usuario_Id', 'Genero', 'AÃ±o_de_nacimiento', 'Inicio_del_viaje', 'Fin_del_viaje', 'Origen_Id', 'Destino_Id']
Columnas en 2026-01: ['Viaje_Id', 'Usuario_Id', 'Genero', 'Año_de_nacimiento', 'Inicio_del_viaje', 'Fin_del_viaje', 'Origen_Id', 'Destino_Id']

Patrones regex de normalización: {'viaje': '^viaje', 'usuario': '^usuario', 'genero': '^genero', 'inicio': '^inicio', 'fin': '^fin', 'origen': '^origen', 'destino': '^destino', 'nacimiento': 'naci'}


### Justificación 2: Eliminación de identificadores innecesarios (`Viaje_Id`, `Usuario_Id`)
- Ni el identificador del viaje ni el del usuario se utilizan para los modelos propuestos (demanda por estación, riesgo operativo o clustering de estaciones).
- Su eliminación ahorra memoria RAM y espacio en disco al guardar en Parquet.

In [7]:
mem_total = df_2025_10.memory_usage(deep=True).sum() / (1024 * 1024)
mem_ids = df_2025_10[["Viaje_Id", "Usuario_Id"]].memory_usage(deep=True).sum() / (1024 * 1024)
print(f"Memoria total en raw: {mem_total:.2f} MB")
print(f"Memoria ocupada por Viaje_Id y Usuario_Id: {mem_ids:.2f} MB ({mem_ids / mem_total:.1%})")

Memoria total en raw: 41.31 MB
Memoria ocupada por Viaje_Id y Usuario_Id: 6.41 MB (15.5%)


### Justificación 3: Fechas de nacimiento nulas
- Existen registros donde el año de nacimiento es nulo (`NaN`).
- Como `NaN` es de tipo flotante en Pandas, toda la columna se ve forzada a `float64`.
- Se eliminan estas filas (menos del 0.05% del dataset) para poder convertir `nacimiento` a tipo `int` sin datos ficticios.

In [ ]:
col_naci = next(c for c in df_2025_10.columns if "naci" in c.lower())
n_null = df_2025_10[col_naci].isna().sum()
print(f"Registros con nacimiento nulo en 2025-10: {n_null} ({n_null / len(df_2025_10):.3%})")
df_2025_10[df_2025_10[col_naci].isna()].head(3)

### Justificación 4: Imputación de valores nulos en Género
- El campo `Genero` contiene valores nulos en aproximadamente el 1% de los viajes.
- En lugar de descartar estas filas y perder información de viajes válidos, se imputan con la categoría `NB` (No Binario / No especificado).

In [9]:
gen_col = next(c for c in df_2025_10.columns if "genero" in c.lower())
n_gen_null = df_2025_10[gen_col].isna().sum()
print(f"Registros con género nulo en 2025-10: {n_gen_null} ({n_gen_null / len(df_2025_10):.2%})")
print("Distribución actual de valores:\n", df_2025_10[gen_col].value_counts(dropna=False))

Registros con género nulo en 2025-10: 4343 (1.03%)
Distribución actual de valores:
 Genero
M      298816
F      116905
NaN      4343
Name: count, dtype: int64


### Justificación 5: Viajes falsos/mecánicos (< 2 min con mismo origen y destino) y duraciones negativas
- Cuando un usuario retira una bicicleta y nota una falla mecánica (o cancela su viaje), la reancla de inmediato en la misma estación en menos de 2 minutos.
- Estos no constituyen viajes reales de transporte y distorsionarían las métricas de demanda por estación.
- Además, se descartan viajes con duración negativa por posibles fallas de sincronización horaria.
- Se descartan segundos al no ser significantes para las problemáticas a resolver.

In [10]:
ini_col = next(c for c in df_2025_10.columns if "inicio" in c.lower())
fin_col = next(c for c in df_2025_10.columns if "fin" in c.lower())
orig_col = next(c for c in df_2025_10.columns if "origen" in c.lower())
dest_col = next(c for c in df_2025_10.columns if "destino" in c.lower())

ini = pd.to_datetime(df_2025_10[ini_col], format="mixed", errors="coerce")
fin = pd.to_datetime(df_2025_10[fin_col], format="mixed", errors="coerce")
dur = (fin - ini).dt.total_seconds() // 60

falsos_viajes = (dur < 2) & (df_2025_10[orig_col] == df_2025_10[dest_col])
negativos = dur < 0

print(f"Viajes < 2 min misma estación (errores al sacar): {falsos_viajes.sum():,} ({falsos_viajes.sum() / len(df_2025_10):.2%})")
print(f"Viajes con duración negativa: {negativos.sum():,}")

Viajes < 2 min misma estación (errores al sacar): 11,436 (2.72%)
Viajes con duración negativa: 0


## 3. Exploración base de otros CSVs del diccionario URLS_VIAJES

Verificamos la carga y exploración base (info y head 10) en otros dos periodos del diccionario: **2026-03** y **2026-09**.

In [11]:
df_2026_03 = load_data_travels(period="2026-03")
df_2026_03.info()

2026-10-17 22:51:02.519 | INFO     | mi_bici.dataset:load_data_travels:161 - Cargando archivo: datos_abiertos_2026_03.csv
<class 'pandas.DataFrame'>
RangeIndex: 376243 entries, 0 to 376242
Data columns (total 8 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   Viaje_Id            376243 non-null  int64  
 1   Usuario_Id          376243 non-null  int64  
 2   Genero              373773 non-null  str    
 3   AÃ±o_de_nacimiento  376204 non-null  float64
 4   Inicio_del_viaje    376243 non-null  str    
 5   Fin_del_viaje       376243 non-null  str    
 6   Origen_Id           376243 non-null  int64  
 7   Destino_Id          376243 non-null  int64  
dtypes: float64(1), int64(4), str(3)
memory usage: 37.0 MB


In [ ]:
df_2026_03.head(10)

In [13]:
df_2026_09 = load_data_travels(period="2026-09")
df_2026_09.info()

2026-10-17 22:51:02.725 | INFO     | mi_bici.dataset:load_data_travels:161 - Cargando archivo: datos_abiertos_2026_09.csv
<class 'pandas.DataFrame'>
RangeIndex: 490056 entries, 0 to 490055
Data columns (total 8 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   Viaje_Id            490056 non-null  int64  
 1   Usuario_Id          490056 non-null  int64  
 2   Genero              486761 non-null  str    
 3   AÃ±o_de_nacimiento  490047 non-null  float64
 4   Inicio_del_viaje    490056 non-null  str    
 5   Fin_del_viaje       490056 non-null  str    
 6   Origen_Id           490056 non-null  int64  
 7   Destino_Id          490056 non-null  int64  
dtypes: float64(1), int64(4), str(3)
memory usage: 48.2 MB


In [ ]:
df_2026_09.head(10)

## 4. Validación de la limpieza con `clean_data_travels`

Aplicamos la función de limpieza implementada en `mi_bici.dataset` sobre uno de los datasets raw para corroborar el resultado final antes de la conversión a Parquet.

In [15]:
df_clean = clean_data_travels(df_2025_10.copy())
df_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 408597 entries, 0 to 408596
Data columns (total 7 columns):
 #   Column        Non-Null Count   Dtype
---  ------        --------------   -----
 0   genero        408597 non-null  str  
 1   nacimiento    408597 non-null  int64
 2   inicio        408597 non-null  str  
 3   fin           408597 non-null  str  
 4   origen        408597 non-null  int64
 5   destino       408597 non-null  int64
 6   duracion_min  408597 non-null  Int64
dtypes: Int64(1), int64(3), str(3)
memory usage: 35.2 MB


In [ ]:
df_clean.head(10)

In [17]:
n_ini = len(df_2025_10)
n_fin = len(df_clean)
diff = n_ini - n_fin
print(f"Filas iniciales en raw: {n_ini:,}")
print(f"Filas después de limpieza: {n_fin:,}")
print(f"Filas descartadas:        {diff:,} ({diff / n_ini:.2%})")
print("\nColumnas limpias:", df_clean.columns.tolist())
print("Tipos de datos resultantes:\n", df_clean.dtypes)

Filas iniciales en raw: 420,064
Filas después de limpieza: 408,597
Filas descartadas:        11,467 (2.73%)

Columnas limpias: ['genero', 'nacimiento', 'inicio', 'fin', 'origen', 'destino', 'duracion_min']
Tipos de datos resultantes:
 genero            str
nacimiento      int64
inicio            str
fin               str
origen          int64
destino         int64
duracion_min    Int64
dtype: object


## Carga de antiguos dataset

Se cargan algunos dataset antiguos en caso de llegar a ser necesarios la limpieza los soporte.

In [ ]:
df_2020_01 = load_data_from_url("https://mibici.net/site/assets/files/1230/datos_abiertos_2020_01.csv")
df_2020_01.head(10)

In [ ]:
df_2023_11 = load_data_from_url("https://mibici.net/site/assets/files/57344/datos_abiertos_2023_11.csv")
df_2023_11.head(10)

In [20]:
ini_col = next(c for c in df_2023_11.columns if "inicio" in c.lower())
fin_col = next(c for c in df_2023_11.columns if "fin" in c.lower())
orig_col = next(c for c in df_2023_11.columns if "origen" in c.lower())
dest_col = next(c for c in df_2023_11.columns if "destino" in c.lower())

ini = pd.to_datetime(df_2023_11[ini_col], format="mixed", dayfirst=True, errors="coerce")
fin = pd.to_datetime(df_2023_11[fin_col], format="mixed", dayfirst=True, errors="coerce")
dur = (fin - ini).dt.total_seconds() // 60

falsos_viajes = (dur < 2) & (df_2023_11[orig_col] == df_2023_11[dest_col])
negativos = dur < 0

print(f"Viajes < 2 min misma estación (errores al sacar): {falsos_viajes.sum():,} ({falsos_viajes.sum() / len(df_2023_11):.2%})")
print(f"Viajes con duración negativa: {negativos.sum():,}")

Viajes < 2 min misma estación (errores al sacar): 14,111 (3.67%)
Viajes con duración negativa: 0


**Nota**: Usar dayfirst=True para evitar que pyton use format US por defecto

## Estadisticas descriptivas de ejemplo

In [21]:
df_2026_08.info()

<class 'pandas.DataFrame'>
RangeIndex: 393524 entries, 0 to 393523
Data columns (total 8 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   Viaje_Id            393524 non-null  int64  
 1   Usuario_Id          393524 non-null  int64  
 2   Genero              391192 non-null  str    
 3   AÃ±o_de_nacimiento  393473 non-null  float64
 4   Inicio_del_viaje    393524 non-null  str    
 5   Fin_del_viaje       393524 non-null  str    
 6   Origen_Id           393524 non-null  int64  
 7   Destino_Id          393524 non-null  int64  
dtypes: float64(1), int64(4), str(3)
memory usage: 38.7 MB


In [22]:
df_2026_08.describe()

,Viaje_Id,Usuario_Id,AÃ±o_de_nacimiento,Origen_Id,Destino_Id
count,3.935240e+05,3.935240e+05,393473.000000,393524.000000,393524.000000
mean,4.548004e+07,2.817606e+06,1990.621626,176.655294,177.097458
std,1.299351e+05,1.848593e+06,15.044138,134.119531,135.192892
min,4.525566e+07,1.020000e+02,199.000000,2.000000,2.000000
25%,4.536767e+07,7.010760e+05,1985.000000,52.000000,51.000000
50%,4.547931e+07,3.401076e+06,1993.000000,165.000000,164.000000
75%,4.559202e+07,4.624554e+06,1999.000000,272.000000,273.000000
max,4.570529e+07,4.770514e+06,2010.000000,519.000000,519.000000


In [23]:
df_2026_08.describe(include="str")

,Genero,Inicio_del_viaje,Fin_del_viaje
count,391192,393524,393524
unique,2,350487,350302
top,M,2026-08-04 14:18:52,2026-08-03 21:25:40
freq,284705,5,6
